# Parametrized Types

Types can be specified as parameters in classes and methods, using the syntax `[T]`.

Before looking into that we can see how we can use these parameters in exsiting generic classes.


### Option types

One example of a type parameter is in generic classes, like `Option`. We cna specify that we want an Option of `Int`:

In [ ]:
val maybeName:Option[Int] = Some("ralph")

-- [E007] Type Mismatch Error: cmd0.sc:1:33 ------------------------------------
1 |val maybeName:Option[Int] = Some("ralph")
  |                                 ^^^^^^^
  |                                 Found:    ("ralph" : String)
  |                                 Required: Int
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

This helps restricting the type of data and values that are acceptable.

In [1]:
val maybeName:Option[String] = Some("ralph")

maybeName: Option[String] = Some(value = "ralph")

### Declaring type parameters in classes

We can declare a class where some types are parametrizable:

In [2]:
case class Person[T](name:String,age:T)

defined class Person

Then when we create instances, they can be using different specific types instead of `T`, like `Int` or `Double`.

In [3]:
val jimmy = Person("jimmy",34)

jimmy: Person[Int] = Person(name = "jimmy", age = 34)

In [4]:
val luigi = Person("luigi",34.5)

luigi: Person[Double] = Person(name = "luigi", age = 34.5)

In [5]:
val jenny = Person("jenny","sixteen")

jenny: Person[String] = Person(name = "jenny", age = "sixteen")

The explicit type parameter helps avoiding inconsistencies:

In [5]:
val lara:Person[Int] = Person("lara",15.6)

-- [E007] Type Mismatch Error: cmd5.sc:1:37 ------------------------------------
1 |val lara:Person[Int] = Person("lara",15.6)
  |                                     ^^^^
  |                                     Found:    (15.6d : Double)
  |                                     Required: Int
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Type parameters restrictions

However sometimes type parameters are too general and do not always make sense. For example age of a `Student`

In [6]:
case class Student[P](age:P)

defined class Student

We can declare instances with any age type:

In [7]:
val s = Student(true)

s: Student[Boolean] = Student(age = true)

In [8]:
val s = Student(List(2,3,4))

s: Student[List[Int]] = Student(age = List(2, 3, 4))

So these exmaples make little sense. We can restrict the age to only be of value data types:

In [9]:
case class Student[T<:AnyVal](age:T)

defined class Student

In [9]:
val s = Student(List(35,6,3))

-- [E007] Type Mismatch Error: cmd9.sc:1:20 ------------------------------------
1 |val s = Student(List(35,6,3))
  |                ^^^^^^^^^^^^
  |Found:    List[Int]
  |Required: AnyVal
  |Note that implicit conversions cannot be applied because they are ambiguous;
  |both method ArrowAssoc in object Predef and method StringFormat in object Predef convert from List[Int] to T
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Parametrized types and inheritance

We can see other effects of type parameters and inheritance. Consider this hierarchy.

In [10]:
trait Athlete

class Swimmer extends Athlete

class Fotballer extends Athlete

defined trait Athlete
defined class Swimmer
defined class Fotballer

We can create a team class with a champion, whose type is parametrizable:

In [11]:
case class SportsTeam[T<:Athlete](champion:T)

defined class SportsTeam

Now if we create instances, the type restrictions prevents form using arbitrary types:

In [11]:
val swimTeam = SportsTeam[String]

-- [E057] Type Mismatch Error: cmd11.sc:1:26 -----------------------------------
1 |val swimTeam = SportsTeam[String]
  |                          ^
  |Type argument String does not conform to upper bound ammonite.$sess.cmd10.wrapper.cmd9.Athlete
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

But it will work for any subtype of `Athlete`:

In [12]:
val alex =  new Swimmer
val swimTeam = SportsTeam[Swimmer](alex)

alex: Swimmer = ammonite.$sess.cmd9$Helper$Swimmer@747ac4df
swimTeam: SportsTeam[Swimmer] = SportsTeam(
  champion = ammonite.$sess.cmd9$Helper$Swimmer@747ac4df
)

The same for a football player:

In [14]:
val jake = new Fotballer
val sionTeam = SportsTeam[Fotballer](jake)

val aTeam = SportsTeam[Athlete](jake)

jake: Fotballer = ammonite.$sess.cmd9$Helper$Fotballer@7ac221d0
sionTeam: SportsTeam[Fotballer] = SportsTeam(
  champion = ammonite.$sess.cmd9$Helper$Fotballer@7ac221d0
)
aTeam: SportsTeam[Athlete] = SportsTeam(
  champion = ammonite.$sess.cmd9$Helper$Fotballer@7ac221d0
)

However, can we cast the sport team with the supertype?

In [13]:
val someTeam:SportsTeam[Athlete] = sionTeam

-- [E007] Type Mismatch Error: cmd13.sc:1:35 -----------------------------------
1 |val someTeam:SportsTeam[Athlete] = sionTeam
  |                                   ^^^^^^^^
  |          Found:    (cmd13.this.cmd12.sionTeam : 
  |            ammonite.$sess.cmd12².wrapper.cmd10.SportsTeam[
  |              ammonite.$sess.cmd12².wrapper.cmd9.Fotballer
  |            ]
  |          )
  |          Required: cmd13.this.cmd10².SportsTeam[cmd13.this.cmd9².Athlete]
  |
  |          where:    cmd10  is a value in class cmd12
  |                    cmd10² is a value in class cmd13
  |                    cmd12  is a value in class cmd13
  |                    cmd12² is a object in package ammonite.$sess
  |                    cmd9   is a value in class cmd12
  |                    cmd9²  is a value in class cmd13
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Covariants

As it was defined, it is an invariant, so the `SportsTeam[Footballer]` is not recognized as subtype of `SportsTeam[Athlete]`.
To fix this we can use covariants, defined using `+` before the type parameter.

In [16]:
case class SportsTeam[+T<:Athlete](champion:T)

defined class SportsTeam

Now the intuitive subtyping is working correctly:

In [17]:
val jake = new Fotballer
val sionTeam = SportsTeam[Fotballer](jake)
val someTeam:SportsTeam[Athlete] = sionTeam

jake: Fotballer = ammonite.$sess.cmd9$Helper$Fotballer@5975f77f
sionTeam: SportsTeam[Fotballer] = SportsTeam(
  champion = ammonite.$sess.cmd9$Helper$Fotballer@5975f77f
)
someTeam: SportsTeam[Athlete] = SportsTeam(
  champion = ammonite.$sess.cmd9$Helper$Fotballer@5975f77f
)

### Type params in methods

We can also define type parameters in methods.

In [18]:
class SportsCompetition[T]


def giveAward[T](competition:SportsCompetition[T], athlete:T) = println ("winner is "+athlete)

defined class SportsCompetition
defined function giveAward

The the type of competition can be defined when using the method.

In [19]:
val swissCup = new SportsCompetition[Fotballer]

val shakiri = new Fotballer

giveAward(swissCup,shakiri)

winner is ammonite.$sess.cmd9$Helper$Fotballer@2d5424f9


swissCup: SportsCompetition[Fotballer] = ammonite.$sess.cmd17$Helper$SportsCompetition@31eccb27
shakiri: Fotballer = ammonite.$sess.cmd9$Helper$Fotballer@2d5424f9

What if we do the same, but in this case the competition is for any athelete?

In [19]:
val swissCup = new SportsCompetition[Athlete]

val shakiri = new Fotballer

giveAward[Fotballer](swissCup,shakiri)

-- [E007] Type Mismatch Error: cmd19.sc:5:35 -----------------------------------
5 |val res19_2 = giveAward[Fotballer](swissCup,shakiri)
  |                                   ^^^^^^^^
  |   Found:    (Helper.this.swissCup : 
  |     cmd19.this.cmd17.SportsCompetition[cmd19.this.cmd9.Athlete]
  |   )
  |   Required: cmd19.this.cmd17.SportsCompetition[cmd19.this.cmd9.Fotballer]
  |
  | longer explanation available when compiling with `-explain`Compilation Failed

: 

### Contravariance

As opposed to covariants, we can solve this problem with contravariants, which are the opposite. We can use the supertype as parameter.
The notation for contravariants is `-`


In [20]:
class SportsCompetition[-T]


def giveAward[T](competition:SportsCompetition[T], athlete:T) = println ("winner is "+athlete)

defined class SportsCompetition
defined function giveAward

Now the award of a Footballer to an Athlete competition is possible.

In [21]:
val swissCup = new SportsCompetition[Athlete]

val shakiri = new Fotballer

giveAward[Fotballer](swissCup,shakiri)

winner is ammonite.$sess.cmd9$Helper$Fotballer@33f86986


swissCup: SportsCompetition[Athlete] = ammonite.$sess.cmd19$Helper$SportsCompetition@45f30a77
shakiri: Fotballer = ammonite.$sess.cmd9$Helper$Fotballer@33f86986